In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
import torch
print(torch.cuda.is_available())

True


In [68]:
!git clone https://github.com/jugal-sac/IR-colorization-BAH2026.git
%cd IR-colorization-BAH2026
!ls

fatal: destination path 'IR-colorization-BAH2026' already exists and is not an empty directory.
/content/IR-colorization-BAH2026/IR-colorization-BAH2026
driver.py  input  output  README.md  scripts  utils


In [69]:
!pip install rasterio numpy opencv-python matplotlib tifffile scikit-image

In [70]:
!pip install earthengine-api geemap

In [71]:
import rasterio, numpy, cv2, matplotlib, tifffile, skimage, ee, geemap
from PIL import Image
print("All dependencies confirmed working")

All dependencies confirmed working


In [72]:
!grep -n "PROJECT_ID\|ee_project_id" scripts/download_data.sh


11:EE_PROJECT_ID="hackathonisro2026"
13:python scripts/download.py $PRODUCT_ID $BANDS $START_DATE $END_DATE $OUTPUT_PATH --ee_project_id $EE_PROJECT_ID


In [73]:
!sed -i 's/YOUR_ACTUAL_PROJECT_ID/hackathonisro2026/g' scripts/download_data.sh
!cat scripts/download_data.sh | grep -i project

EE_PROJECT_ID="hackathonisro2026"
python scripts/download.py $PRODUCT_ID $BANDS $START_DATE $END_DATE $OUTPUT_PATH --ee_project_id $EE_PROJECT_ID


In [74]:
import geemap
help(geemap.ee_export_image)

Help on function ee_export_image in module geemap.common:

ee_export_image(ee_object: ee.image.Image, filename: str, scale: float | None = None, crs: str | None = None, crs_transform: list[float] | None = None, region: typing.Any | None = None, dimensions: list[int] | None = None, file_per_band: bool = False, format: str = 'ZIPPED_GEO_TIFF', unzip: bool = True, unmask_value: float | None = None, timeout: int = 300, proxies: dict[str, typing.Any] | None = None, verbose: bool = True) -> None
    Exports an ee.Image as a GeoTIFF.

    Args:
        ee_object: The ee.Image to download.
        filename: Output filename for the exported image.
        scale: A default scale to use for any bands that do not specify one; ignored if
            crs and crs_transform is specified.
        crs: A default CRS string to use for any bands that do not explicitly specify
            one.
        crs_transform: a default affine transform to use for any bands that do not
            specify one, of the

In [75]:
!grep -n "filename_prefix" scripts/download.py

26:        filename_prefix = f'landsat9_{product_id}'


In [76]:
!grep -n "filename_prefix\|ee_export_image" scripts/download.py

26:        filename_prefix = f'landsat9_{product_id}'
29:            geemap.ee_export_image(image, filename=output_path, scale=30, region=image.geometry().bounds(), file_per_band=True)


In [77]:
file_path = 'scripts/download.py'

with open(file_path, 'r') as f:
    content = f.read()

old_line = "geemap.ee_export_image(image, output_path, scale=30, region=image.geometry().bounds(), file_per_band=True, filename_prefix=filename_prefix)"
new_line = "geemap.ee_export_image(image, filename=output_path, scale=30, region=image.geometry().bounds(), file_per_band=True)"

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(file_path, 'w') as f:
        f.write(content)
    print("✅ Line replaced successfully")
else:
    print("⚠️ Exact line not found — paste the grep output so we match it precisely")

⚠️ Exact line not found — paste the grep output so we match it precisely


In [78]:
!grep -n "ee_export_image" scripts/download.py

29:            geemap.ee_export_image(image, filename=output_path, scale=30, region=image.geometry().bounds(), file_per_band=True)


In [79]:
!grep -n "filename_prefix" scripts/download.py

26:        filename_prefix = f'landsat9_{product_id}'


In [80]:
!./scripts/download_data.sh

Starting data download...
The filename must end with .tif


In [81]:
import ee

# This forces the link to appear inside the notebook output
ee.Authenticate(auth_mode='notebook')


To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=VGR0347JWkssO5hjyflWJvZUu7J_25o0M_Mfm_LaU6E&tc=OBtDxq9JdirsFvCW1t-m50JUQqyAEs4yS1CPtoByVvE&cc=92RyWcAYH9Z3ZzMKA1BEa5ht5dS0LioNh8ozP4bjzrk

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPyiXqw6X4eMCmTrHko3VcY5n8N1F9nntqEcSlBmYUnh-H61v2LZHH4

Successfully saved authorization token.


In [82]:
import ee
ee.Initialize(project='hackathonisro2026')
print(ee.String('Connected!').getInfo())

Connected!


In [83]:
# Fix the hardcoded placeholder project ID in the script
!sed -i 's/your-google-earth-engine-project-id/hackathonisro2026/g' \
    scripts/download_data.sh

# Run it
!chmod +x scripts/download_data.sh
!./scripts/download_data.sh

Starting data download...
The filename must end with .tif


In [84]:
import ee
import os

ee.Initialize(project='hackathonisro2026')

# Create output folder
os.makedirs('input/demo_product', exist_ok=True)

# Load Landsat 9 - pick a scene over India
image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
    .filterDate('2024-01-01', '2024-03-31') \
    .filterBounds(ee.Geometry.Point([77.2090, 28.6139])) \
    .sort('CLOUD_COVER') \
    .first()

print('Scene found:', image.get('LANDSAT_PRODUCT_ID').getInfo())

Scene found: LC09_L1TP_146040_20240229_20240229_02_T1


In [85]:
import geemap

# Define area (Delhi region - change if needed)
aoi = ee.Geometry.Rectangle([76.8, 28.4, 77.6, 29.0])

bands = {
    'B2': 'input/demo_product/scene_B2.tif',
    'B3': 'input/demo_product/scene_B3.tif',
    'B4': 'input/demo_product/scene_B4.tif',
    'B10': 'input/demo_product/scene_B10.tif',
}

for band, path in bands.items():
    print(f'Downloading {band}...')
    geemap.ee_export_image(
        image.select([band]),
        filename=path,
        scale=30,
        region=aoi,
        file_per_band=False
    )
    print(f'✅ {band} saved to {path}')

Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/IR-colorization-BAH2026/input/demo_product/scene_B2.tif
✅ B2 saved to input/demo_product/scene_B2.tif
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/IR-colorization-BAH2026/input/demo_product/scene_B3.tif
✅ B3 saved to input/demo_product/scene_B3.tif
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/IR-colorization-BAH2026/input/demo_product/scene_B4.tif
✅ B4 saved to input/demo_product/scene_B4.tif
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/IR-colorization-BAH2026/input/demo_product/scene_B10.tif
✅ B10 saved to input/demo_product/scene_B10.tif


In [86]:
import rasterio

for band, path in bands.items():
    with rasterio.open(path) as src:
        print(f'✅ {band} | Resolution: {src.res} | Shape: {src.shape}')

✅ B2 | Resolution: (30.0, 30.0) | Shape: (2266, 2648)
✅ B3 | Resolution: (30.0, 30.0) | Shape: (2266, 2648)
✅ B4 | Resolution: (30.0, 30.0) | Shape: (2266, 2648)
✅ B10 | Resolution: (30.0, 30.0) | Shape: (2266, 2648)


In [87]:
!pip install rasterio tifffile opencv-python-headless \
    scikit-image numpy pillow --quiet
print("✅ Dependencies installed")

✅ Dependencies installed


In [88]:
import os
os.chdir('/content/IR-colorization-BAH2026')
!ls  # confirm driver.py is here

driver.py  input  IR-colorization-BAH2026  output  README.md  scripts  utils


In [89]:
import os, shutil

# driver.py looks for files ending in _B2.TIF _B3.TIF _B4.TIF _B10.TIF
# Your files are named scene_B2.tif etc — need to rename

src_dir = 'input/demo_product/'
files = os.listdir(src_dir)
print("Current files:", files)

for f in files:
    if f.startswith('scene_'):
        # scene_B2.tif → demo_product_B2.TIF
        band = f.split('_')[1].replace('.tif', '')
        new_name = f'demo_scene_{band}.TIF'
        os.rename(os.path.join(src_dir, f),
                  os.path.join(src_dir, new_name))
        print(f"Renamed: {f} → {new_name}")

print("\nFiles after rename:")
print(os.listdir(src_dir))

Current files: ['scene_B3.tif', 'demo_product_B10.TIF', 'demo_scene_B4.TIF', 'demo_scene_B2.TIF', 'demo_scene_B10.TIF', 'demo_product_B3.TIF', 'demo_scene_B3.TIF', 'scene_B4.tif', 'scene_B2.tif', 'demo_product_B2.TIF', 'scene_B10.tif', 'demo_product_B4.TIF']
Renamed: scene_B3.tif → demo_scene_B3.TIF
Renamed: scene_B4.tif → demo_scene_B4.TIF
Renamed: scene_B2.tif → demo_scene_B2.TIF
Renamed: scene_B10.tif → demo_scene_B10.TIF

Files after rename:
['demo_product_B10.TIF', 'demo_scene_B4.TIF', 'demo_scene_B2.TIF', 'demo_scene_B10.TIF', 'demo_product_B3.TIF', 'demo_scene_B3.TIF', 'demo_product_B2.TIF', 'demo_product_B4.TIF']


In [90]:
import os
os.chdir('/content/IR-colorization-BAH2026')
print("Current dir:", os.getcwd())
print("Contents:", os.listdir('.'))

Current dir: /content/IR-colorization-BAH2026
Contents: ['IR-colorization-BAH2026', '.git', 'README.md', 'output', 'scripts', 'driver.py', 'utils', 'input']


In [91]:
import os
print("Contents of /content:", os.listdir('/content'))

Contents of /content: ['.config', 'IR-colorization-BAH2026', 'drive', 'sample_data']


In [92]:
import os
os.chdir('/content')

!git clone https://github.com/DarkCipher-z/bah2026-ps10-team.git IR-colorization-BAH2026

if os.path.isdir('/content/IR-colorization-BAH2026'):
    os.chdir('/content/IR-colorization-BAH2026')
    print("✅ Now in:", os.getcwd())
    print("Contents:", os.listdir('.'))
else:
    print("❌ Clone failed - check repo URL")

fatal: destination path 'IR-colorization-BAH2026' already exists and is not an empty directory.
✅ Now in: /content/IR-colorization-BAH2026
Contents: ['IR-colorization-BAH2026', '.git', 'README.md', 'output', 'scripts', 'driver.py', 'utils', 'input']


In [93]:
import os
print(os.listdir('/content/drive/MyDrive/BAH2026'))

[' tir2rgb_pipeline.ipynb', 'input', 'output', 'datasets']


In [94]:
!find /content/drive/MyDrive -iname "*.git" -maxdepth 5 2>/dev/null

In [95]:
import shutil
shutil.copytree('/content/drive/MyDrive/BAH2026/input/', '/content/IR-colorization-BAH2026/input/', dirs_exist_ok=True)
shutil.copytree('/content/drive/MyDrive/BAH2026/output/', '/content/IR-colorization-BAH2026/output/', dirs_exist_ok=True)
print("✅ input/ and output/ restored from Drive")

✅ input/ and output/ restored from Drive


In [96]:
import os

if os.path.exists('input/demo_product/'):
    print("✅ Found:", os.listdir('input/demo_product/'))
else:
    print("❌ Folder missing - session reset, need to re-download")

✅ Found: ['scene_B3.tif', 'demo_product_B10.TIF', 'demo_scene_B4.TIF', 'demo_scene_B2.TIF', 'demo_scene_B10.TIF', 'demo_product_B3.TIF', 'demo_scene_B3.TIF', 'scene_B4.tif', 'scene_B2.tif', 'demo_product_B2.TIF', 'scene_B10.tif', 'demo_product_B4.TIF']


In [97]:
# Re-download bands from Earth Engine
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')
os.makedirs('input/demo_product/', exist_ok=True)

image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
    .filterDate('2024-01-01', '2024-03-31') \
    .filterBounds(ee.Geometry.Point([77.2090, 28.6139])) \
    .sort('CLOUD_COVER').first()

aoi = ee.Geometry.Rectangle([76.8, 28.4, 77.6, 29.0])

for band, path in {
    'B2': 'input/demo_product/scene_B2.TIF',
    'B3': 'input/demo_product/scene_B3.TIF',
    'B4': 'input/demo_product/scene_B4.TIF',
    'B10': 'input/demo_product/scene_B10.TIF'
}.items():
    print(f'Downloading {band}...')
    geemap.ee_export_image(
        image.select([band]),
        filename=path, scale=30,
        region=aoi, file_per_band=False
    )
    print(f'✅ {band} done')

Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/demo_product/scene_B2.TIF
✅ B2 done
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/demo_product/scene_B3.TIF
✅ B3 done
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/demo_product/scene_B4.TIF
✅ B4 done
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/demo_product/scene_B10.TIF
✅ B10 done


In [98]:
import os
os.chdir('/content/IR-colorization-BAH2026')
!ls  # confirm driver.py is here

driver.py  input  IR-colorization-BAH2026  output  README.md  scripts  utils


In [99]:
import os

src_dir = 'input/demo_product/'
files = os.listdir(src_dir)
print("Before rename:", files)

for f in files:
    if f.startswith('scene_') and f.endswith('.TIF'):
        # scene_B2.TIF → demo_scene_B2.TIF
        band = f.replace('scene_', '').replace('.TIF', '')
        new_name = f'demo_scene_{band}.TIF'
        os.rename(
            os.path.join(src_dir, f),
            os.path.join(src_dir, new_name)
        )
        print(f"✅ {f} → {new_name}")

print("\nAfter rename:", os.listdir(src_dir))

Before rename: ['scene_B3.tif', 'demo_product_B10.TIF', 'demo_scene_B4.TIF', 'demo_scene_B2.TIF', 'demo_scene_B10.TIF', 'demo_product_B3.TIF', 'demo_scene_B3.TIF', 'scene_B4.tif', 'scene_B2.tif', 'demo_product_B2.TIF', 'scene_B10.tif', 'demo_product_B4.TIF']

After rename: ['scene_B3.tif', 'demo_product_B10.TIF', 'demo_scene_B4.TIF', 'demo_scene_B2.TIF', 'demo_scene_B10.TIF', 'demo_product_B3.TIF', 'demo_scene_B3.TIF', 'scene_B4.tif', 'scene_B2.tif', 'demo_product_B2.TIF', 'scene_B10.tif', 'demo_product_B4.TIF']


In [100]:
!pip install tifffile opencv-python-headless \
    scikit-image pillow --quiet
print("✅ Done")

✅ Done


In [101]:
!python driver.py

2026-06-27 18:57:38,958 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 18:57:38,959 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 18:57:39,617 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 18:57:39,617 - /content

In [102]:
import os

patch_dir = 'output/patches/'
if os.path.exists(patch_dir):
    files = os.listdir(patch_dir)
    npy = [f for f in files if f.endswith('.npy')]
    png = [f for f in files if f.endswith('.png')]
    print(f"✅ .npy patches: {len(npy)}")
    print(f"📸 .png previews: {len(png)}")
    print("Sample:", files[:4])
else:
    print("❌ No patches folder - check driver.py output above")

✅ .npy patches: 0
📸 .png previews: 0
Sample: ['jaipur', 'aurangabad', 'demo', 'vadodara']


In [103]:
import os

# Walk all subfolders to find patches
for root, dirs, files in os.walk('output/'):
    for f in files:
        print(os.path.join(root, f))

output/patches.log
output/patches/jaipur/sample_004/rgb_100m_512.png
output/patches/jaipur/sample_004/rgb_100m_512.npy
output/patches/jaipur/sample_004/tir_200m.png
output/patches/jaipur/sample_004/tir_100m_512.png
output/patches/jaipur/sample_004/tir_100m_512.npy
output/patches/jaipur/sample_004/tir_200m.npy
output/patches/jaipur/sample_005/rgb_100m_512.png
output/patches/jaipur/sample_005/rgb_100m_512.npy
output/patches/jaipur/sample_005/tir_200m.png
output/patches/jaipur/sample_005/tir_100m_512.png
output/patches/jaipur/sample_005/tir_100m_512.npy
output/patches/jaipur/sample_005/tir_200m.npy
output/patches/jaipur/sample_002/rgb_100m_512.png
output/patches/jaipur/sample_002/rgb_100m_512.npy
output/patches/jaipur/sample_002/tir_200m.png
output/patches/jaipur/sample_002/tir_100m_512.png
output/patches/jaipur/sample_002/tir_100m_512.npy
output/patches/jaipur/sample_002/tir_200m.npy
output/patches/jaipur/sample_003/rgb_100m_512.png
output/patches/jaipur/sample_003/rgb_100m_512.npy
outpu

In [104]:
import shutil, os

base = '/content/drive/MyDrive/BAH2026'

# Save patches
shutil.copytree(
    'output/',
    f'{base}/output/',
    dirs_exist_ok=True
)
# Save raw bands
shutil.copytree(
    'input/',
    f'{base}/input/',
    dirs_exist_ok=True
)
print("✅ Everything saved to Drive!")

✅ Everything saved to Drive!


In [105]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Different locations across India for varied terrain
scenes_config = [
    ([72.8, 18.9, 73.2, 19.3], 'mumbai_coast'),
    ([77.5, 12.8, 78.0, 13.2], 'bengaluru_urban'),
    ([88.2, 22.4, 88.6, 22.8], 'kolkata_river'),
    ([78.0, 26.0, 78.5, 26.5], 'gwalior_rural'),
    ([76.5, 30.5, 77.0, 31.0], 'chandigarh_mixed'),
]

for coords, name in scenes_config:
    print(f"\n📥 Downloading {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/scene_{band}.TIF'
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,
            region=aoi,
            file_per_band=False
        )
    print(f"✅ {name} done!")

print("\n🎉 All scenes downloaded!")


📥 Downloading mumbai_coast...
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/mumbai_coast/scene_B2.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/mumbai_coast/scene_B3.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/mumbai_coast/scene_B4.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/mumbai_coast/scene_B10.TIF
✅ mumbai_coast done!

📥 Downloading bengaluru_urban...
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/bengaluru_urban/scene_B2.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/bengaluru_urban/scene_B3.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/bengaluru_urban/scene_B4.TIF
Generating URL ...
Please wait ...
Data downloaded to /content/IR-color

In [106]:
import os

input_dir = 'input/'
for scene in os.listdir(input_dir):
    scene_path = os.path.join(input_dir, scene)
    if not os.path.isdir(scene_path):
        continue
    for f in os.listdir(scene_path):
        if f.startswith('scene_') and \
           (f.endswith('.TIF') or f.endswith('.tif')):
            band = f.replace('scene_', '') \
                    .replace('.tif','').replace('.TIF','')
            new_name = f'{scene}_{band}.TIF'
            os.rename(
                os.path.join(scene_path, f),
                os.path.join(scene_path, new_name)
            )
            print(f"✅ {f} → {new_name}")

print("\nAll scenes ready!")

✅ scene_B3.tif → demo_product_B3.TIF
✅ scene_B4.tif → demo_product_B4.TIF
✅ scene_B2.tif → demo_product_B2.TIF
✅ scene_B10.tif → demo_product_B10.TIF
✅ scene_B3.tif → mumbai_coast_B3.TIF
✅ scene_B4.tif → mumbai_coast_B4.TIF
✅ scene_B2.tif → mumbai_coast_B2.TIF
✅ scene_B10.tif → mumbai_coast_B10.TIF
✅ scene_B3.tif → chandigarh_mixed_B3.TIF
✅ scene_B4.tif → chandigarh_mixed_B4.TIF
✅ scene_B2.tif → chandigarh_mixed_B2.TIF
✅ scene_B10.tif → chandigarh_mixed_B10.TIF
✅ scene_B3.tif → bengaluru_urban_B3.TIF
✅ scene_B4.tif → bengaluru_urban_B4.TIF
✅ scene_B2.tif → bengaluru_urban_B2.TIF
✅ scene_B10.tif → bengaluru_urban_B10.TIF
✅ scene_B3.tif → kolkata_river_B3.TIF
✅ scene_B4.tif → kolkata_river_B4.TIF
✅ scene_B2.tif → kolkata_river_B2.TIF
✅ scene_B10.tif → kolkata_river_B10.TIF
✅ scene_B3.tif → gwalior_rural_B3.TIF
✅ scene_B4.tif → gwalior_rural_B4.TIF
✅ scene_B2.tif → gwalior_rural_B2.TIF
✅ scene_B10.tif → gwalior_rural_B10.TIF

All scenes ready!


In [107]:
!python driver.py

2026-06-27 19:01:50,973 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:01:50,974 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:01:51,642 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:01:51,642 - /content

In [108]:
import os

total_npy = 0
total_png = 0

for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    png = [f for f in files if f.endswith('.png')]
    if npy or png:
        print(f"📁 {root}: {len(npy)} npy, {len(png)} png")
    total_npy += len(npy)
    total_png += len(png)

print(f"\n{'='*40}")
print(f"Total .npy patches: {total_npy}")
print(f"Total .png previews: {total_png}")
if total_npy >= 100:
    print("✅ Enough patches to start training!")
else:
    print("⚠️ Need more scenes — add more in Step 3")

📁 output/patches/jaipur/sample_004: 3 npy, 3 png
📁 output/patches/jaipur/sample_005: 3 npy, 3 png
📁 output/patches/jaipur/sample_002: 3 npy, 3 png
📁 output/patches/jaipur/sample_003: 3 npy, 3 png
📁 output/patches/jaipur/sample_001: 3 npy, 3 png
📁 output/patches/jaipur/sample_000: 3 npy, 3 png
📁 output/patches/aurangabad/sample_004: 3 npy, 3 png
📁 output/patches/aurangabad/sample_005: 3 npy, 3 png
📁 output/patches/aurangabad/sample_002: 3 npy, 3 png
📁 output/patches/aurangabad/sample_003: 3 npy, 3 png
📁 output/patches/aurangabad/sample_001: 3 npy, 3 png
📁 output/patches/aurangabad/sample_000: 3 npy, 3 png
📁 output/patches/demo/sample_004: 3 npy, 3 png
📁 output/patches/demo/sample_005: 3 npy, 3 png
📁 output/patches/demo/sample_002: 3 npy, 3 png
📁 output/patches/demo/sample_006: 3 npy, 3 png
📁 output/patches/demo/sample_003: 3 npy, 3 png
📁 output/patches/demo/sample_001: 3 npy, 3 png
📁 output/patches/demo/sample_000: 3 npy, 3 png
📁 output/patches/vadodara/sample_004: 3 npy, 3 png
📁 output

In [109]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# MUCH larger areas (1.5° × 1.5°) = ~165km × 165km
# This will give hundreds of patches per scene
scenes_config = [
    ([74.0, 17.5, 75.5, 19.0], 'maharashtra'),
    ([76.5, 11.5, 78.0, 13.0], 'karnataka'),
    ([86.5, 21.5, 88.0, 23.0], 'westbengal'),
    ([76.5, 29.0, 78.0, 30.5], 'haryana'),
    ([79.5, 25.0, 81.0, 26.5], 'uttarpradesh'),
    ([72.5, 22.5, 74.0, 24.0], 'gujarat'),
]

for coords, name in scenes_config:
    print(f"\n📥 Downloading {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)

    # Get least cloudy scene
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-04-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene found for {name}, skipping")
        continue

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} already exists")
            continue
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,
            region=aoi,
            file_per_band=False
        )
        print(f"  ✅ {band} done")

print("\n🎉 All scenes downloaded!")


📥 Downloading maharashtra...
  Scene: LC09_L1TP_146048_20240401_20240401_02_T1
Generating URL ...
An error occurred while downloading.
Total request size (88251507 bytes) must be less than or equal to 50331648 bytes.
  ✅ B2 done
Generating URL ...
An error occurred while downloading.
Total request size (88251507 bytes) must be less than or equal to 50331648 bytes.
  ✅ B3 done
Generating URL ...
An error occurred while downloading.
Total request size (88251507 bytes) must be less than or equal to 50331648 bytes.
  ✅ B4 done
Generating URL ...
An error occurred while downloading.
Total request size (88251507 bytes) must be less than or equal to 50331648 bytes.
  ✅ B10 done

📥 Downloading karnataka...
  Scene: LC09_L1TP_145051_20240206_20240206_02_T1
Generating URL ...
An error occurred while downloading.
Total request size (91869393 bytes) must be less than or equal to 50331648 bytes.
  ✅ B2 done
Generating URL ...
An error occurred while downloading.
Total request size (91869393 bytes)

In [110]:
import shutil, os

# Clear old tiny patches
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)

print("✅ Old output cleared")
print("Now running driver.py...")
!python driver.py

✅ Old output cleared
Now running driver.py...
2026-06-27 19:04:45,501 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:04:45,502 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:04:45,968 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 1

In [111]:
import os

total_npy = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)} patches")
    total_npy += len(npy)

print(f"\n{'='*40}")
print(f"Total .npy patches: {total_npy}")
if total_npy >= 500:
    print("✅ Good enough to start training!")
elif total_npy >= 100:
    print("⚠️ Decent start - can begin smoke test")
else:
    print("❌ Still too few - areas too small")

📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches
📁 sample_000: 3 patches

Total .npy patches: 51
❌ Still too few - areas too small


In [112]:
import os
import rasterio

print("Checking downscaled image sizes:")
downscale_dir = 'output/downscaled_data/'

for f in os.listdir(downscale_dir):
    if f.endswith('.tif'):
        with rasterio.open(os.path.join(downscale_dir, f)) as src:
            print(f"{f}: {src.width}x{src.height} pixels")

Checking downscaled image sizes:
lucknow_rgb_100m.tif: 797x667 pixels
chennai_tir_100m.tif: 869x666 pixels
aurangabad_tir_100m.tif: 841x669 pixels
tn_tile1_rgb_100m.tif: 333x335 pixels
tn_tile3_tir_100m.tif: 333x335 pixels
visakhapatnam_rgb_100m.tif: 859x677 pixels
raj_tile1_tir_200m.tif: 155x169 pixels
tn_tile5_tir_100m.tif: 332x334 pixels
tn_tile2_tir_100m.tif: 334x336 pixels
vadodara_tir_200m.tif: 414x338 pixels
vadodara_rgb_100m.tif: 829x677 pixels
raj_tile2_tir_100m.tif: 308x336 pixels
patna_tir_100m.tif: 813x675 pixels
bengaluru_urban_rgb_100m.tif: 548x450 pixels
raj_tile2_rgb_100m.tif: 308x336 pixels
tn_tile3_rgb_100m.tif: 333x335 pixels
raj_tile5_tir_100m.tif: 306x336 pixels
bhubaneswar_rgb_100m.tif: 838x669 pixels
raj_tile5_rgb_100m.tif: 306x336 pixels
tn_tile2_tir_200m.tif: 167x168 pixels
indore_tir_100m.tif: 827x671 pixels
vadodara_tir_100m.tif: 829x677 pixels
indore_rgb_100m.tif: 827x671 pixels
chennai_rgb_100m.tif: 869x666 pixels
guwahati_rgb_100m.tif: 804x671 pixels
patna

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [113]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Very large area - entire Rajasthan region
# 5° × 5° = ~550km × 550km - will give thousands of patches
large_scenes = [
    ([72.0, 24.0, 77.0, 29.0], 'rajasthan'),   # 5°×5°
    ([77.0, 8.0,  82.0, 13.0], 'tamilnadu'),    # 5°×5°
]

for coords, name in large_scenes:
    print(f"\n📥 Downloading {name} (large area)...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene ID: {pid}")
    except:
        print(f"  ⚠️ No scene for {name}")
        continue

    for band in ['B2', 'B3', 'B4', 'B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        print(f"  Downloading {band}...")
        geemap.ee_export_image(
            image.select([band]),
            filename=path,
            scale=30,        # 30m native resolution
            region=aoi,
            file_per_band=False
        )
        print(f"  ✅ {band} done")

print("\n🎉 Large scene download complete!")


📥 Downloading rajasthan (large area)...
  Scene ID: LC09_L1TP_146041_20240519_20240519_02_T1
Generating URL ...
An error occurred while downloading.
Total request size (942944856 bytes) must be less than or equal to 50331648 bytes.
  ✅ B2 done
Generating URL ...
An error occurred while downloading.
Total request size (942944856 bytes) must be less than or equal to 50331648 bytes.
  ✅ B3 done
Generating URL ...
An error occurred while downloading.
Total request size (942944856 bytes) must be less than or equal to 50331648 bytes.
  ✅ B4 done
Generating URL ...
An error occurred while downloading.
Total request size (942944856 bytes) must be less than or equal to 50331648 bytes.
  ✅ B10 done

📥 Downloading tamilnadu (large area)...
  Scene ID: LC09_L1TP_144051_20240505_20240505_02_T1
Generating URL ...
An error occurred while downloading.
Total request size (1043824518 bytes) must be less than or equal to 50331648 bytes.
  ✅ B2 done
Generating URL ...
An error occurred while downloading.

In [114]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Split large area into smaller 1°×1° tiles at 100m scale
# This stays under the 50MB limit
scenes = [
    ([72.0, 24.0, 73.0, 25.0], 'raj_tile1'),
    ([73.0, 24.0, 74.0, 25.0], 'raj_tile2'),
    ([74.0, 24.0, 75.0, 25.0], 'raj_tile3'),
    ([72.0, 25.0, 73.0, 26.0], 'raj_tile4'),
    ([73.0, 25.0, 74.0, 26.0], 'raj_tile5'),
    ([77.0, 8.0,  78.0, 9.0],  'tn_tile1'),
    ([78.0, 8.0,  79.0, 9.0],  'tn_tile2'),
    ([77.0, 9.0,  78.0, 10.0], 'tn_tile3'),
    ([78.0, 9.0,  79.0, 10.0], 'tn_tile4'),
    ([79.0, 9.0,  80.0, 10.0], 'tn_tile5'),
]

for coords, name in scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)

    aoi = ee.Geometry.Rectangle(coords)
    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=100,  # 100m = smaller file size
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as ex:
            print(f"  ❌ {band}: {ex}")

print("\n✅ All done!")


📥 raj_tile1...
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile1/raj_tile1_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile1/raj_tile1_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile1/raj_tile1_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile1/raj_tile1_B10.TIF
  ✅ B10

📥 raj_tile2...
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile2/raj_tile2_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile2/raj_tile2_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/raj_tile2/raj_tile2_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorizatio

In [115]:
import shutil, os

shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
print("✅ Cleared")

!python driver.py


✅ Cleared
2026-06-27 19:07:53,689 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:07:53,690 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:07:54,271 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:07:54,271 

In [116]:
import os

total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)

print(f"\nTotal patches: {total}")
print("✅ Enough!" if total >= 200
      else "⚠️ Run more tiles")

📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3
📁 sample_000: 3

Total patches: 51
⚠️ Run more tiles


In [117]:
import shutil, os

base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)

# Save patches
shutil.copytree('output/',
    f'{base}/output/', dirs_exist_ok=True)

# Save raw bands
shutil.copytree('input/',
    f'{base}/input/', dirs_exist_ok=True)

print("✅ Everything saved to Drive!")

✅ Everything saved to Drive!


In [118]:
# Read the file and find patch size
with open('scripts/create_patches.py', 'r') as f:
    content = f.read()

# Show lines with patch size
for i, line in enumerate(content.split('\n')):
    if '256' in line or 'patch' in line.lower() \
       or 'size' in line.lower():
        print(f"{i+1}: {line}")

44: def create_patches(input_root, output_root):
88:         logger.info(f"Generating full dataset patches for {product_id}...")
91:         for y in range(0, h200 - 256 + 1, 256):
92:             for x in range(0, w200 - 256 + 1, 256):
93:                 patch_200m_tir = tir_200m[..., y:y+256, x:x+256]
96:                 patch_100m_tir_512 = tir_100m[..., y100:y100+512, x100:x100+512]
97:                 patch_100m_rgb_512 = rgb_100m[..., y100:y100+512, x100:x100+512]
99:                 if patch_100m_tir_512.shape[-2:] != (512, 512) or patch_100m_rgb_512.shape[-2:] != (512, 512):
107:                     'tir_200m': patch_200m_tir,
108:                     'tir_100m_512': patch_100m_tir_512,
109:                     'rgb_100m_512': patch_100m_rgb_512
121:     parser = argparse.ArgumentParser(description='Create co-registered image patches.')
123:     parser.add_argument('--output_dir', type=str, default='output/patches', help='Path to output directory.')
125:     create_patches(arg

In [119]:
with open('scripts/create_patches.py', 'r') as f:
    content = f.read()

# Replace 256 with 64 everywhere
new_content = content.replace('256', '64')

with open('scripts/create_patches.py', 'w') as f:
    f.write(new_content)

print("✅ Patch size changed to 64×64")

# Verify
!grep -n "64\|patch_size" scripts/create_patches.py \
    | head -10

✅ Patch size changed to 64×64
91:        for y in range(0, h200 - 64 + 1, 64):
92:            for x in range(0, w200 - 64 + 1, 64):
93:                patch_200m_tir = tir_200m[..., y:y+64, x:x+64]


In [120]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 19:10:16,213 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:10:16,214 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:10:16,770 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:10:16,770 - /content

In [121]:
import os
total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)
print(f"\nTotal: {total}")
print("✅ Good!" if total >= 200 else "⚠️ Need more")

📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample

In [122]:
import shutil, os
base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)
shutil.copytree('output/', f'{base}/output/',
                dirs_exist_ok=True)
shutil.copytree('input/', f'{base}/input/',
                dirs_exist_ok=True)
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [123]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# Same size as demo (Delhi) which worked well
# 0.8° × 0.6° areas at scale=30
new_scenes = [
    ([80.8, 26.7, 81.6, 27.3], 'lucknow'),
    ([85.0, 25.3, 85.8, 25.9], 'patna'),
    ([73.7, 19.8, 74.5, 20.4], 'aurangabad'),
    ([75.7, 26.8, 76.5, 27.4], 'jaipur'),
    ([76.9, 10.9, 77.7, 11.5], 'coimbatore'),
]

for coords, name in new_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skip")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,  # 30m like demo!
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 lucknow...
  Scene: LC09_L1TP_144041_20240505_20240505_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/lucknow/lucknow_B10.TIF
  ✅ B10

📥 patna...
  Scene: LC09_L1TP_141042_20240430_20240430_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/patna/patna_B4.TIF
  ✅ B4
Generating URL ...
Plea

In [124]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 19:15:38,445 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:15:38,446 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:15:39,092 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:15:39,093 - /content

In [125]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

# 0.8° x 0.6° boxes at scale=30 — same recipe that gave 6 patches each
more_scenes = [
    ([75.7, 22.6, 76.5, 23.2], 'indore'),
    ([83.2, 17.6, 84.0, 18.2], 'visakhapatnam'),
    ([91.7, 26.1, 92.5, 26.7], 'guwahati'),
    ([70.7, 22.9, 71.5, 23.5], 'rajkot'),
    ([81.5, 21.0, 82.3, 21.6], 'raipur'),
]

for coords, name in more_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skipping")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 indore...
  Scene: LC09_L1TP_146044_20240503_20240503_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/indore/indore_B10.TIF
  ✅ B10

📥 visakhapatnam...
  Scene: LC09_L1TP_141047_20240329_20240329_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visakhapatnam_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visakhapatnam_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/visakhapatnam/visa

In [126]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 19:20:50,771 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:20:50,772 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:20:51,219 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:20:51,220 - /content

In [127]:
import ee, geemap, os

ee.Initialize(project='hackathonisro2026')

more_scenes = [
    ([77.2, 28.4, 78.0, 29.0], 'noida'),
    ([72.5, 23.0, 73.3, 23.6], 'vadodara'),
    ([80.2, 13.0, 81.0, 13.6], 'chennai'),
    ([75.3, 19.8, 76.1, 20.4], 'nashik'),
    ([85.8, 20.2, 86.6, 20.8], 'bhubaneswar'),
]

for coords, name in more_scenes:
    print(f"\n📥 {name}...")
    os.makedirs(f'input/{name}/', exist_ok=True)
    aoi = ee.Geometry.Rectangle(coords)

    image = ee.ImageCollection('LANDSAT/LC09/C02/T1') \
        .filterDate('2024-01-01', '2024-06-30') \
        .filterBounds(aoi) \
        .sort('CLOUD_COVER') \
        .first()

    try:
        pid = image.get('LANDSAT_PRODUCT_ID').getInfo()
        print(f"  Scene: {pid}")
    except:
        print(f"  ⚠️ No scene, skipping")
        continue

    for band in ['B2','B3','B4','B10']:
        path = f'input/{name}/{name}_{band}.TIF'
        if os.path.exists(path):
            print(f"  ⏭️ {band} exists")
            continue
        try:
            geemap.ee_export_image(
                image.select([band]),
                filename=path,
                scale=30,
                region=aoi,
                file_per_band=False
            )
            print(f"  ✅ {band}")
        except Exception as e:
            print(f"  ❌ {band}: {e}")

print("\n✅ Done!")


📥 noida...
  Scene: LC09_L1TP_146041_20240519_20240519_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B4.TIF
  ✅ B4
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/noida/noida_B10.TIF
  ✅ B10

📥 vadodara...
  Scene: LC09_L1TP_148044_20240501_20240501_02_T1
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B2.TIF
  ✅ B2
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B3.TIF
  ✅ B3
Generating URL ...
Please wait ...
Data downloaded to /content/IR-colorization-BAH2026/input/vadodara/vadodara_B4.TIF
  ✅ B4
Generating URL ...
P

In [128]:
import shutil, os
shutil.rmtree('output/', ignore_errors=True)
os.makedirs('output/', exist_ok=True)
!python driver.py

2026-06-27 19:25:52,737 - /content/IR-colorization-BAH2026/output - INFO - Processing product: tn_tile2
2026-06-27 19:25:52,738 - /content/IR-colorization-BAH2026/output - INFO - Running: python /content/IR-colorization-BAH2026/scripts/merge_rgb.py /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B4.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B3.tif /content/IR-colorization-BAH2026/input/tn_tile2/tn_tile2_B2.tif /content/IR-colorization-BAH2026/output/rgb_images/tn_tile2_rgb_30m.tif
2026-06-27 19:25:53,185 - /content/IR-colorization-BAH2026/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-06-27 19:25:53,185 - /content

In [129]:
import os
total = 0
for root, dirs, files in os.walk('output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    if npy:
        print(f"📁 {os.path.basename(root)}: {len(npy)}")
    total += len(npy)
print(f"\nTotal: {total}")
print("✅ Proceed to Pix2Pix!" if total >= 100 else "⚠️ Add more")

📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample_002: 3
📁 sample_003: 3
📁 sample_001: 3
📁 sample_000: 3
📁 sample_004: 3
📁 sample_005: 3
📁 sample

In [130]:
import shutil, os
base = '/content/drive/MyDrive/BAH2026'
os.makedirs(base, exist_ok=True)
shutil.copytree('output/', f'{base}/output/', dirs_exist_ok=True)
shutil.copytree('input/', f'{base}/input/', dirs_exist_ok=True)
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [131]:
!git config --global user.email "sanketaher2329@gmail.com"
!git config --global user.name "DarkCipher-z"
print("✅ Git configured")

✅ Git configured


In [132]:
import shutil, os

src = '/content/drive/MyDrive/BAH2026/tir2rgb_pipeline.ipynb'
dst = f'/content/{REPO_NAME}/tir2rgb_pipeline.ipynb'

if os.path.exists(src):
    shutil.copy(src, dst)
    print("✅ Notebook copied")
else:
    print("⚠️ Notebook not found at", src)
    print("Files in BAH2026 folder:", os.listdir('/content/drive/MyDrive/BAH2026'))

NameError: name 'REPO_NAME' is not defined

In [ ]:
os.chdir(f'/content/{REPO_NAME}')

!git config --global user.email "your@email.com"
!git config --global user.name "DarkCipher-z"
!git add .
!git commit -m "Day 1: Data download + patch generation complete"

push_result = os.system("git push origin main")
if push_result == 0:
    print("✅ Pushed to GitHub!")
else:
    print("❌ Push failed - see error above")

In [ ]:
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-iname', '*tir2rgb*'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import os, shutil
from google.colab import userdata

# === Step 1: Define everything fresh (required every new session) ===
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER  = "DarkCipher-z"
REPO_NAME    = "bah2026-ps10-team"

# === Step 2: Re-clone repo (this is gone every time the runtime resets) ===
os.chdir('/content')
if os.path.isdir(f'/content/{REPO_NAME}'):
    shutil.rmtree(f'/content/{REPO_NAME}')  # clean slate, avoid "already exists" errors

clone_result = os.system(f"git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git")
if not os.path.isdir(f'/content/{REPO_NAME}'):
    raise SystemExit("❌ Clone failed - check token/repo access before continuing")
print("✅ Repo cloned")

# === Step 3: Copy the notebook (confirmed real path from your Drive) ===
src = '/content/drive/MyDrive/BAH2026/ tir2rgb_pipeline.ipynb'
dst = f'/content/{REPO_NAME}/ tir2rgb_pipeline.ipynb'

if os.path.exists(src):
    shutil.copy(src, dst)
    print("✅ Notebook copied")
else:
    raise SystemExit(f"❌ Notebook still not found at {src}")

# === Step 4: Commit and push ===
os.chdir(f'/content/{REPO_NAME}')

os.system('git config --global user.email "sanketaher2329@gmail.com"')
os.system('git config --global user.name "DarkCipher-z"')
os.system('git add .')
os.system('git commit -m "Day 1: Data download + patch generation complete"')

push_result = os.system("git push origin main")
if push_result == 0:
    print("✅ Pushed to GitHub!")
else:
    print("❌ Push failed - see error above")

In [ ]:
import os
os.chdir('/content')
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
os.chdir('pytorch-CycleGAN-and-pix2pix')
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix ready!")

In [ ]:
import os

print("Before:", os.getcwd())

target = '/content/pytorch-CycleGAN-and-pix2pix'
if os.path.isdir(target):
    os.chdir(target)
    print("After:", os.getcwd())
    print("Files here:", os.listdir('.')[:10])
else:
    print("❌ Folder doesn't exist - clone may have gone elsewhere. Check /content:")
    print(os.listdir('/content'))

In [ ]:
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix ready!")

In [ ]:
import os
nested = '/content/pytorch-CycleGAN-and-pix2pix/IR-colorization-BAH2026'
print("Contents:", os.listdir(nested))

In [ ]:
import shutil, os

src = '/content/pytorch-CycleGAN-and-pix2pix/IR-colorization-BAH2026'
dst = '/content/IR-colorization-BAH2026'

if os.path.isdir(dst):
    shutil.rmtree(dst)  # clear out the empty/incomplete one from earlier

shutil.move(src, dst)
print("✅ Moved to:", dst)
print("Contents:", os.listdir(dst))

In [ ]:
shutil.copytree('/content/drive/MyDrive/BAH2026/input/', f'{dst}/input/', dirs_exist_ok=True)
shutil.copytree('/content/drive/MyDrive/BAH2026/output/', f'{dst}/output/', dirs_exist_ok=True)
print("✅ input/ and output/ restored")

os.chdir(dst)
print("Now in:", os.getcwd())
print("Top-level contents:", os.listdir('.'))

In [ ]:
import os
total = 0
for root, dirs, files in os.walk('/content/IR-colorization-BAH2026/output/patches/'):
    npy = [f for f in files if f.endswith('.npy')]
    total += len(npy)
print(f"Total patches restored: {total}")

In [ ]:
import os
os.chdir('/content/pytorch-CycleGAN-and-pix2pix')
print("Now in:", os.getcwd())
print("Contents:", os.listdir('.'))

In [ ]:
!pip install -r requirements.txt --quiet
print("✅ Pix2Pix dependencies installed")

In [ ]:
!pip install dominate visdom wandb --quiet
print("✅ Pix2Pix dependencies installed")

In [ ]:
!cat environment.yml

In [ ]:
import numpy as np
from PIL import Image
import os

os.makedirs('datasets/tir2rgb/train', exist_ok=True)
os.makedirs('datasets/tir2rgb/val', exist_ok=True)

patch_base = '/content/IR-colorization-BAH2026/output/patches/'
count = 0

for scene in os.listdir(patch_base):
    scene_path = os.path.join(patch_base, scene)
    if not os.path.isdir(scene_path):
        continue
    for sample in os.listdir(scene_path):
        sp = os.path.join(scene_path, sample)
        if not os.path.isdir(sp):
            continue
        files = os.listdir(sp)
        tir_f = next((f for f in files if 'tir_200m' in f and f.endswith('.npy')), None)
        rgb_f = next((f for f in files if 'rgb' in f and f.endswith('.npy')), None)
        if not tir_f or not rgb_f:
            continue

        tir = np.load(os.path.join(sp, tir_f))
        rgb = np.load(os.path.join(sp, rgb_f))

        def norm(x):
            x = x.astype(np.float32)
            mn, mx = x.min(), x.max()
            if mx - mn < 1e-8:
                return np.zeros_like(x, dtype=np.uint8)
            return ((x - mn) / (mx - mn) * 255).astype(np.uint8)

        tir_n = norm(tir.squeeze())
        tir_rgb = np.stack([tir_n]*3, axis=-1)

        if rgb.ndim == 3 and rgb.shape[0] == 3:
            rgb_img = np.transpose(rgb, (1,2,0))
        else:
            rgb_img = rgb
        rgb_img = norm(rgb_img)
        if rgb_img.ndim == 2:
            rgb_img = np.stack([rgb_img]*3, axis=-1)

        tir_pil = Image.fromarray(tir_rgb).resize((256,256))
        rgb_pil = Image.fromarray(rgb_img).resize((256,256))

        combined = Image.new('RGB', (512, 256))
        combined.paste(tir_pil, (0, 0))
        combined.paste(rgb_pil, (256, 0))

        folder = 'val' if count % 10 == 0 else 'train'
        combined.save(f'datasets/tir2rgb/{folder}/{count:04d}.png')
        count += 1

print(f"✅ {count} training pairs created!")
print(f"Train: {len(os.listdir('datasets/tir2rgb/train'))} | Val: {len(os.listdir('datasets/tir2rgb/val'))}")

In [ ]:
import torch
if torch.cuda.is_available():
    print("✅ GPU ready - starting training")
    !python train.py \
        --dataroot datasets/tir2rgb \
        --name tir2rgb_smoke \
        --model pix2pix \
        --direction AtoB \
        --n_epochs 10 \
        --batch_size 4 \
        --gpu_ids 0 \
        --display_id 0 \
        --save_epoch_freq 5
else:
    print("⚠️ No GPU - check Runtime > Change runtime type > GPU")

In [ ]:
!pip install dominate visdom wandb --quiet
print("✅ Pix2Pix dependencies installed")

In [ ]:
import os
print("Current dir:", os.getcwd())
print("Contents:", os.listdir('.'))

In [ ]:
import os
target = '/content/pytorch-CycleGAN-and-pix2pix'
if os.path.isdir(target):
    os.chdir(target)
    print("✅ Now in:", os.getcwd())
else:
    print("❌ Folder missing - runtime likely reset, re-clone needed")

In [ ]:
import os
print("=== LIVE CHECK ===")
print("Current dir:", os.getcwd())
print("Contents:", os.listdir('.'))
print("requirements.txt exists?", os.path.exists('requirements.txt'))
print("environment.yml exists?", os.path.exists('environment.yml'))

In [ ]:
!pip install dominate visdom wandb --quiet

In [ ]:
import shutil

# Back up training pairs to Drive so future disconnects don't cost you this work again
shutil.copytree('datasets/tir2rgb', '/content/drive/MyDrive/BAH2026/datasets/tir2rgb', dirs_exist_ok=True)
print("✅ Training pairs backed up to Drive")

import os
print("Backed up files:", len(os.listdir('/content/drive/MyDrive/BAH2026/datasets/tir2rgb/train')))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

samples = sorted(os.listdir('datasets/tir2rgb/train'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, fname in zip(axes.flat, samples):
    img = Image.open(f'datasets/tir2rgb/train/{fname}')
    ax.imshow(img)
    ax.set_title(fname)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
import os
print("Current dir:", os.getcwd())

In [ ]:
import shutil
shutil.copytree('datasets/tir2rgb', '/content/drive/MyDrive/BAH2026/datasets/tir2rgb', dirs_exist_ok=True)
print("✅ Training pairs backed up to Drive - safe even if runtime resets again")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

samples = sorted(os.listdir('datasets/tir2rgb/train'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, fname in zip(axes.flat, samples):
    img = Image.open(f'datasets/tir2rgb/train/{fname}')
    ax.imshow(img)
    ax.set_title(fname)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Save this as a script for later - don't run training, just confirm it's syntactically valid
test_script = '''
import torch
print("GPU check:", torch.cuda.is_available())
'''
with open('/content/gpu_check.py', 'w') as f:
    f.write(test_script)
print("✅ Saved gpu_check.py - run this first when GPU returns")